In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
stores = spark.table("workspace.supermarket.raw_stores")

products = spark.table("workspace.supermarket.raw_products")

suppliers = spark.table("workspace.supermarket.raw_suppliers")

sales = spark.table(
    "workspace.supermarket.raw_sales_transactions"
)

inventory = spark.table(
    "workspace.supermarket.raw_inventory"
)

purchase_orders = spark.table(
    "workspace.supermarket.raw_purchase_orders"
)

In [0]:
stores.select(
    [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in stores.columns
    ]
).show()

In [0]:
display(
    stores.filter(
        F.col("manager_name").isNull()
    )
)

In [0]:
clean_stores = stores \
    .dropDuplicates(["store_id"]) \
    .withColumn(
        "manager_name",
        F.coalesce(
            F.col("manager_name"),
            F.lit("Unknown")
        )
    )

In [0]:
display(clean_stores)

In [0]:
clean_stores.filter(
    F.col("manager_name").isNull()
).count()

In [0]:
display(
    products.filter(
        F.col("unit_price") <= 0
    )
)

In [0]:
valid_products = products.filter(
    F.col("unit_price") > 0
)

category_medians = valid_products.groupBy(
    "category"
).agg(
    F.expr(
        "percentile_approx(unit_price, 0.5)"
    ).alias("median_price")
)

display(category_medians)

In [0]:
clean_products = products.join(
    category_medians,
    on="category",
    how="left"
).withColumn(
    "unit_price",
    F.when(
        F.col("unit_price") <= 0,
        F.col("median_price")
    ).otherwise(
        F.col("unit_price")
    )
).drop("median_price")

In [0]:
display(
    clean_products.filter(
        F.col("unit_price") <= 0
    )
)

In [0]:
display(
    suppliers
    .filter(
        ~F.col("phone").rlike(r"^\d{10}$")
    )
)

In [0]:
clean_suppliers = suppliers.withColumn(
    "phone",
    F.regexp_replace(
        F.col("phone"),
        r"[^0-9]",
        ""
    )
)

In [0]:
clean_suppliers = clean_suppliers.withColumn(
    "phone",
    F.when(
        F.length("phone") == 12,
        F.substring("phone", 3, 10)
    ).otherwise(
        F.col("phone")
    )
)

In [0]:
display(clean_suppliers)

In [0]:
display(
    clean_suppliers.filter(
        ~F.col("phone").rlike(r"^\d{10}$")
    )
)

In [0]:
duplicate_sales = sales.groupBy(
    "store_id",
    "product_id",
    "quantity_sold",
    "unit_price",
    "total_amount",
    "sale_date"
).count().filter(
    F.col("count") > 1
)

display(duplicate_sales)

In [0]:
clean_sales = sales.dropDuplicates([
    "store_id",
    "product_id",
    "quantity_sold",
    "unit_price",
    "total_amount",
    "sale_date"
])

In [0]:
print("Raw sales:", sales.count())
print("Clean sales:", clean_sales.count())

In [0]:
clean_sales = clean_sales.withColumn(
    "sale_date",
    F.to_date("sale_date")
)

In [0]:
display(
    clean_sales.filter(
        (F.col("sale_date") < F.lit("2024-01-01")) |
        (F.col("sale_date") > F.lit("2024-12-31"))
    )
)

In [0]:
invalid_store_sales = sales.join(
    stores.select("store_id"),
    on="store_id",
    how="left_anti"
)

display(invalid_store_sales)

In [0]:
invalid_product_sales = sales.join(
    products.select("product_id"),
    on="product_id",
    how="left_anti"
)

display(invalid_product_sales)

In [0]:
display(
    inventory.filter(
        F.col("quantity_on_hand") < 0
    )
)

In [0]:
display(
    inventory.filter(
        F.col("quantity_on_hand") == 0
    )
)

In [0]:
clean_inventory = inventory \
    .dropDuplicates(["inventory_id"]) \
    .filter(
        F.col("quantity_on_hand") >= 0
    )

In [0]:
clean_purchase_orders = purchase_orders \
    .withColumn(
        "order_date",
        F.to_date("order_date")
    ) \
    .withColumn(
        "delivery_date",
        F.to_date("delivery_date")
    )

In [0]:
display(
    clean_purchase_orders.filter(
        F.col("delivery_date").isNull()
    )
)

In [0]:
clean_purchase_orders = clean_purchase_orders.dropDuplicates(
    ["po_id"]
)

In [0]:
clean_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.supermarket.stores"
    )

clean_products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.supermarket.products"
    )

clean_suppliers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.supermarket.suppliers"
    )

clean_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.supermarket.sales_transactions"
    )

clean_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.supermarket.inventory"
    )

clean_purchase_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.supermarket.purchase_orders"
    )

In [0]:
%sql
SHOW TABLES IN workspace.supermarket;

In [0]:
quality_summary = [
    ("stores", "missing_manager_name", 1, "Replaced NULL with 'Unknown'"),
    ("products", "invalid_unit_price", 2, "Replaced with category median"),
    ("suppliers", "inconsistent_phone_format", 2, "Standardized to 10-digit format"),
    ("sales_transactions", "duplicate_records", 5, "Removed duplicate transaction copies"),
    ("sales_transactions", "invalid_dates", 0, "No invalid dates found"),
    ("sales_transactions", "invalid_store_ids", 0, "No invalid store references found"),
    ("sales_transactions", "invalid_product_ids", 0, "No invalid product references found"),
    ("inventory", "negative_quantity", 0, "No negative quantities found"),
    ("inventory", "out_of_stock", 0, "Business condition; retained"),
    ("purchase_orders", "missing_delivery_date", 0, "Pending orders; retained")
]

In [0]:
quality_df = spark.createDataFrame(
    quality_summary,
    [
        "table_name",
        "issue_type",
        "issue_count",
        "action_taken"
    ]
)

display(quality_df)

In [0]:
out_of_stock_count = inventory.filter(
    F.col("quantity_on_hand") == 0
).count()

print("Out-of-stock records:", out_of_stock_count)